# Classmate AI — Master Agent (Composed Notebook)Single source of truth for the **AI / agent layer**, composed for easy debugging and editing. Sections:1. **LLM setup** — per-user Groq + settings2. **Agent tools** — incl. `post_to_telegram`, `generate_assignment_solution`, `generate_study_guide`3. **Master agent graph** — LangGraph, persistent per-student conversation memory4. **Assignment / exam solvers** — auto-solution + study guide (RAG-grounded)5. **Missed-class from outline** — fuses outline + Telegram + RAG

---## 0. Path setup

In [ ]:
import sys
from pathlib import Path

# Make the `app` backend package importable no matter where the notebook is opened.
CWD = Path.cwd()
candidates = [
    CWD,
    CWD.parent,
    CWD / "backend",
    CWD.parent / "backend",
    Path.cwd().resolve() / "ClassMate X" / "backend",
]
for c in candidates:
    if c and c.is_dir() and (c / "app").is_dir() and str(c) not in sys.path:
        sys.path.insert(0, str(c))


---## 1. Imports

In [ ]:
from sqlalchemy.orm import Session

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from app import config, prompt_loader, security
from app.database import SessionLocal
from app.models import ChatHistory, UserSettings, AcademicEvent, SourcePool
from app.knowledge import rag
from app.services import academic, memory
from app import looputil


---## 2. LLM setup

In [ ]:
def get_or_create_settings(db: Session, user_id: int) -> UserSettings:
    """Fetch the user's settings row, creating an empty one if needed."""
    settings = db.query(UserSettings).filter_by(user_id=user_id).first()
    if settings is None:
        settings = UserSettings(user_id=user_id)
        db.add(settings)
        db.commit()
        db.refresh(settings)
    return settings


def get_decrypted_settings(db: Session, user_id: int) -> dict:
    row = get_or_create_settings(db, user_id)
    return {
        "telegram_bot_token": security.decrypt_secret(row.telegram_bot_token),
        "telegram_api_id": security.decrypt_secret(row.telegram_api_id),
        "telegram_api_hash": security.decrypt_secret(row.telegram_api_hash),
        "groq_api_key": security.decrypt_secret(row.groq_api_key),
        "target_chat": row.target_chat or "",
        "bot_enabled": row.bot_enabled,
        "notifications_enabled": row.notifications_enabled,
        "ocr_screenshots": row.ocr_screenshots,
        "poll_after_class": row.poll_after_class,
    }


def save_settings(db: Session, user_id: int, settings: dict) -> UserSettings:
    """Persist credentials encrypted. Empty strings leave values unchanged."""
    row = get_or_create_settings(db, user_id)

    def _maybe(v):
        return v if v is not None else ""

    def _save(field, token):
        if token and "••••" not in token:
            setattr(row, field, security.encrypt_secret(token))

    _save("telegram_bot_token", _maybe(settings.get("telegram_bot_token")))
    _save("telegram_api_id", _maybe(settings.get("telegram_api_id")))
    _save("telegram_api_hash", _maybe(settings.get("telegram_api_hash")))
    _save("groq_api_key", _maybe(settings.get("groq_api_key")))

    for field in ("target_chat", "bot_enabled", "notifications_enabled",
                  "ocr_screenshots", "poll_after_class"):
        if field in settings and settings[field] is not None:
            setattr(row, field, settings[field])

    db.commit()
    db.refresh(row)
    return row


def get_groq_api_key(db: Session, user_id: int) -> str:
    return get_decrypted_settings(db, user_id)["groq_api_key"]


def get_llm(db: Session, user_id: int, model: str | None = None):
    """ChatGroq bound to the user's own key. None when no key configured."""
    key = get_groq_api_key(db, user_id)
    if not key:
        return None
    return ChatGroq(
        model_name=model or config.DEFAULT_LLM_MODEL,
        groq_api_key=key,
        temperature=0.2,
    )


---## 3. Agent tools

In [ ]:
def _send_telegram(user_id: int, chat: str, text: str) -> bool:
    """Fire a Telegram send on the main loop (safe from any thread)."""
    from app.telegram import bot_manager

    looputil.call_coro(bot_manager.send_message(user_id, chat, text))
    return True


def _record_outgoing(user_id: int, chat: str, text: str) -> None:
    db = SessionLocal()
    try:
        academic.add_telegram_message(db, user_id, chat, text, sender_id="classmate_ai_bot")
    finally:
        db.close()


def build_tools(user_id: int) -> list:
    """Tools the master agent can call. Each decider is the LLM itself — tools
    return facts, never hardcoded replies. The agent decides whether/how to act."""

    def _db():
        return SessionLocal()

    @tool
    def search_assignments(query: str) -> str:
        """Search the student's assignments by keyword (course, title, status)."""
        db = _db()
        try:
            rows = academic.list_events(db, user_id, etype="assignment", limit=50)
            return _fmt(rows, default="No assignments found.")
        finally:
            db.close()

    @tool
    def search_deadlines(query: str) -> str:
        """Deadlines grouped by overdue / due today / due soon / upcoming."""
        db = _db()
        try:
            buckets = academic.deadline_buckets(db, user_id)
            lines = []
            for phase in ("overdue", "due_today", "due_soon", "upcoming"):
                group = buckets.get(phase, [])
                if group:
                    lines.append(f"**{phase.replace('_', ' ').title()}:**")
                    for r in group:
                        lines.append(f"- {r['title'] or r['course']} ({r['course']}) — {r['deadline']}")
            return "\n".join(lines) if lines else "No deadlines found."
        finally:
            db.close()

    @tool
    def search_quizzes(query: str) -> str:
        """Quizzes and exams with dates, scope and any stored study guide."""
        db = _db()
        try:
            quiz = academic.list_events(db, user_id, etype="quiz", limit=30)
            exam = academic.list_events(db, user_id, etype="exam", limit=30)
            return _fmt(quiz + exam, default="No quizzes or exams found.")
        finally:
            db.close()

    @tool
    def search_schedule(query: str) -> str:
        """Class schedule and any schedule changes."""
        db = _db()
        try:
            classes = academic.list_events(db, user_id, etype="class", limit=50)
            changes = academic.list_events(db, user_id, etype="schedule_change", limit=20)
            return _fmt(classes + changes, default="No schedule found.")
        finally:
            db.close()

    @tool
    def search_announcements(query: str) -> str:
        """Recent announcements from the group."""
        db = _db()
        try:
            rows = academic.list_events(db, user_id, etype="announcement", limit=20)
            return _fmt(rows, default="No announcements found.")
        finally:
            db.close()

    @tool
    def search_missed(query: str) -> str:
        """Missed-class summaries (what a student missed, topic + key points)."""
        db = _db()
        try:
            rows = academic.list_missed(db, user_id)
            if not rows:
                return "No missed-class summaries yet."
            return "\n\n".join(
                f"[{m.date} {m.course}] {m.topic}\n{m.summary}\n(basis: {m.basis})"
                for m in rows
            )
        finally:
            db.close()

    @tool
    def generate_missed_summary(topic: str, course: str = "", date: str = "",
                                context: str = "") -> str:
        """Build a missed-class summary for a class the student missed / that the
        group says was covered (e.g. 'summarize the last class about transformers').
        Cross-checks the topic against the attached course outline, saves the
        summary to the missed-class dashboard, and returns it. Args: topic (what
        the class covered), course (optional), date (optional YYYY-MM-DD or
        'today'), context (optional extra info the student knows).
        """
        from datetime import date as _date
        from ..services import coverage as cov

        db = _db()
        try:
            c = (course or "").strip()
            if not c:
                c = cov.guess_missed_course(db, user_id, topic)
            c = c or "General"
            d = (date or "").strip()
            if not d or d.lower() in ("today", "yesterday"):
                d = cov.last_class_date(db, user_id, c) or _date.today().isoformat()
            basis = "assistant request" + (f" · course '{c}'" if course else "")
            summary, cross = cov.build_missed_summary(
                db, user_id, c, topic, d, basis, context=context, save=True,
            )
            verdict = ("matches the course outline" if cross.get("matched")
                       else "not found in the course outline")
            item = f" (outline item: {cross['item']})" if cross.get("item") else ""
            return (f"Saved to the missed-class dashboard.\nCourse: {c} · {d}\n"
                    f"Outline cross-check: {verdict}{item}\n\n{summary}")
        finally:
            db.close()

    @tool
    def search_course_docs(query: str) -> str:
        """Vector search across the student's ingested course materials (RAG)."""
        results = rag.search_all_courses(user_id, query, k_per_course=3)
        if not results:
            return "No relevant course material found for this query."
        return "\n\n".join(results[:8])

    @tool
    def student_progress(query: str) -> str:
        """High-level academic progress overview."""
        db = _db()
        try:
            p = academic.progress_summary(db, user_id)
            return (
                f"Assignments total {p['assignments_total']}, open {p['assignments_open']}, "
                f"done {p['assignments_done']} ({p['completed_pct']}% complete). "
                f"Quizzes {p['quizzes']}, exams {p['exams']}, classes {p['classes']}. "
                f"Courses: {', '.join(p['courses']) or 'none'}."
            )
        finally:
            db.close()

    @tool
    def post_to_telegram(message: str) -> str:
        """Post a message to the student's Telegram group. USE when you need to
        ask the class for clarification (e.g. an unclear deadline, exam portion,
        or topic), remind about a deadline, or share an announcement. Decide
        yourself whether posting is warranted; explain in your reply what you
        posted and why."""
        from app.telegram import bot_manager

        db = _db()
        try:
            settings = get_decrypted_settings(db, user_id)
            # Prefer the ACTUAL group where academic messages arrive so the bot
            # posts in the real school group, falling back to target_chat.
            chat = academic.resolve_group(db, user_id, fallback=settings.get("target_chat", ""))
            if not chat:
                return "Cannot post: no Telegram group detected. Make sure the bot is in the school group, or set a target chat in Setup."
            looputil.call_coro(bot_manager.send_message(user_id, chat, message))
            academic.add_telegram_message(db, user_id, chat, message, sender_id="classmate_ai_bot")
            return "Posted to the group."
        finally:
            db.close()

    @tool
    def ask_group(question: str) -> str:
        """Post a short clarifying question to the student's Telegram group to
        confirm uncertain academic information. Use when in doubt."""
        db = _db()
        try:
            settings = get_decrypted_settings(db, user_id)
            chat = academic.resolve_group(db, user_id, fallback=settings.get("target_chat", ""))
            if not chat:
                return "Cannot ask the group: no Telegram group detected. Make sure the bot is in the school group, or set a target chat in Setup."
            text = f"❓ *Clarification wanted* — {question}"
            ok = _send_telegram(user_id, chat, text)
            _record_outgoing(user_id, chat, text)
            return "Sent to the group." if ok else "Bot is offline; question NOT sent."
        finally:
            db.close()

    @tool
    def deploy_group_poll(question: str, options: str, course: str = "General") -> str:
        """Post a native Telegram POLL to the school group to ask the students a
        question (e.g. what today's lecture covered, when an assignment is due,
        or which portion an exam covers). options is a list separated by '|' or
        newlines (2-8 choices). The poll closes once at least 3 students answer;
        the top-voted option wins and (for lecture topics) a study summary is
        generated from the course materials and delivered in the app. Prefer this
        over ask_group when a clear multiple-choice question exists; keep
        ask_group for free-text replies."""
        text = (options or "").replace("\n", "|")
        opts = [o.strip() for o in text.split("|") if o.strip()][:8]
        if len(opts) < 2:
            return "I need at least 2 options to create a poll."

        async def _run():
            from app.telegram import polling
            db = SessionLocal()
            try:
                return await polling.deploy_poll_for_group(db, user_id, course, question, opts)
            finally:
                db.close()

        try:
            poll_id = looputil.run_coro(_run(), timeout=30)
        except Exception as exc:  # noqa: BLE001
            return f"Could not post the poll: {exc}"

        if not poll_id:
            return ("Cannot post the poll: no Telegram group detected or the bot is offline. "
                    "Make sure the bot is in the school group, or set a target chat in Setup.")
        return (f"Poll posted to the school group (id {poll_id}). Students will answer in Telegram; "
                "as soon as 3 vote, the top option wins and I'll add a study summary in the app.")

    @tool
    def remember_user_rule(statement: str) -> str:
        """Learn a persistent fact or preference about the user/community."""
        db = _db()
        try:
            kind = memory.remember_from_statement(db, user_id, statement)
            return f"Remembered [{kind}]: {statement}"
        finally:
            db.close()

    @tool
    def generate_assignment_solution(event_id: int) -> str:
        """Generate (or regenerate) a step-by-step AI solution and short summary
        for a stored assignment, grounded in the course material. Stores the
        result on the assignment. Returns a confirmation + summary."""
        db = SessionLocal()
        try:
            row = db.query(AcademicEvent).filter_by(id=event_id, user_id=user_id, type="assignment").first()
            if row is None:
                return "Assignment not found for this user."
            text = row.description or row.title or ""
            details = row.details or {}
            context = rag.search_all_courses(user_id, text, k_per_course=3)
            solution, summary = _solve_assignment_llm(db, user_id, row, text, context)
            details["summary"] = summary
            details["solution"] = solution
            row.details = details
            db.commit()
            return f"Solved. {summary}"
        finally:
            db.close()

    @tool
    def generate_study_guide(event_id: int) -> str:
        """Generate a study guide + practice questions for a quiz/exam, grounded
        in course material and outline. Stores it on the event. Returns a short
        confirmation."""
        db = SessionLocal()
        try:
            row = db.query(AcademicEvent).filter_by(id=event_id, user_id=user_id).first()
            if row is None or row.type not in ("quiz", "exam"):
                return "Event not found, or not a quiz/exam."
            guide = _study_guide_llm(db, user_id, row)
            details = row.details or {}
            details["study_guide"] = guide
            row.details = details
            db.commit()
            return "Study guide generated."
        finally:
            db.close()

    return [
        search_assignments,
        search_deadlines,
        search_quizzes,
        search_schedule,
        search_announcements,
        search_missed,
        search_course_docs,
        student_progress,
        post_to_telegram,
        ask_group,
        deploy_group_poll,
        remember_user_rule,
        generate_assignment_solution,
        generate_study_guide,
        generate_missed_summary,
    ]


def _fmt(rows, default: str = "Nothing found.") -> str:
    if not rows:
        return default
    lines = []
    for r in rows[:25]:
        head = f"**{r.title or r.course}** ({r.course}) [{r.source}/{r.confidence}]"
        extra = r.deadline or r.event_date or r.topic or ""
        if extra:
            head += f" — {extra}"
        lines.append(head)
        if r.description:
            lines.append(f"  {r.description[:200]}")
        if (r.details or {}).get("summary"):
            lines.append(f"  summary: {(r.details or {})['summary'][:160]}")
    return "\n".join(lines)


---## 4. Master agent graph (LangGraph + memory)

In [ ]:
log = __import__("logging").getLogger("classmate.agent")


def _system_prompt(user_id: int) -> str:
    db = SessionLocal()
    try:
        rules = memory.rules_text(db, user_id)
    finally:
        db.close()
    return prompt_loader.load_prompt("master_agent.txt").format(user_memory=rules)


def build_graph(user_id: int, model: str | None = None):
    if model and model not in config.LLM_MODELS:
        model = None
    db = SessionLocal()
    try:
        model = get_llm(db, user_id, model)
    finally:
        db.close()
    if model is None:
        return _null_graph()
    tools = build_tools(user_id)
    model = model.bind_tools(tools)
    system_prompt = _system_prompt(user_id)

    def agent_node(state: MessagesState):
        msgs = [SystemMessage(content=system_prompt)] + state["messages"]
        return {"messages": [model.invoke(msgs)]}

    graph = StateGraph(MessagesState)
    graph.add_node("agent", agent_node)
    graph.add_node("tools", ToolNode(tools))
    graph.add_edge(START, "agent")
    graph.add_conditional_edges("agent", tools_condition, {"tools": "tools", END: END})
    graph.add_edge("tools", "agent")
    return graph.compile()


def _null_graph():
    def node(state: MessagesState):
        return {
            "messages": [
                AIMessage(
                    content="\u26a0\ufe0f Your Groq API key is not configured. "
                            "Open Settings and add it to unlock the assistant."
                )
            ]
        }

    g = StateGraph(MessagesState)
    g.add_node("agent", node)
    g.add_edge(START, "agent")
    g.add_edge("agent", END)
    return g.compile()


def _load_db_history(db, user_id: int, limit: int) -> list[dict]:
    rows = (
        db.query(ChatHistory)
        .filter_by(user_id=user_id)
        .order_by(ChatHistory.id.desc())
        .limit(limit)
        .all()
    )
    return [{"role": r.role, "content": r.content} for r in reversed(rows)]


def answer(db, user_id: int, message: str,
           history: list[dict] | None = None,
           model: str | None = None) -> str:
    """Ask the master agent a question grounded in the user's data.
    Memoized per student: if the caller does not pass history, the last N turns
    are pulled from the DB (survives refresh/restart) and injected so the agent
    has conversation context. Responses are stored back to DB.
    """
    llm = get_llm(db, user_id)
    if llm is None:
        return _no_llm_reply(db, user_id, message)
    graph = build_graph(user_id, model)
    msgs: list = []
    ctx = history if history is not None else _load_db_history(db, user_id, 10)
    for item in (ctx or [])[-10:]:
        role = item.get("role")
        content = item.get("content", "")
        if role == "assistant":
            msgs.append(AIMessage(content=content))
        else:
            msgs.append(HumanMessage(content=content))
    msgs.append(HumanMessage(content=message))
    try:
        result = graph.invoke(
            {"messages": msgs},
            config={"configurable": {"thread_id": f"user_{user_id}"}},
        )
        reply = result["messages"][-1].content
    except Exception as exc:  # noqa: BLE001
        log.warning("Agent invoke failed user=%s: %s", user_id, exc)
        return f"\u26a0\ufe0f I hit an error while answering: {str(exc)[:200]}"
    db.add(ChatHistory(user_id=user_id, role="user", content=message))
    db.add(ChatHistory(user_id=user_id, role="assistant", content=reply))
    db.commit()
    return reply


def _no_llm_reply(db, user_id: int, message: str) -> str:
    try:
        return (
            "\u26a0\ufe0f No Groq API key configured, so I can't reason. "
            "Add your Groq key in Settings and ask me again."
        )
    finally:
        db.add(ChatHistory(user_id=user_id, role="user", content=message))
        db.commit()


def recent_history(db, user_id: int, limit: int = 30) -> list[dict]:
    rows = (
        db.query(ChatHistory)
        .filter_by(user_id=user_id)
        .order_by(ChatHistory.id.desc())
        .limit(limit)
        .all()
    )
    return [{"role": r.role, "content": r.content} for r in reversed(rows)]


def clear_history(db, user_id: int) -> int:
    """Delete the student's conversation history. Returns count removed."""
    n = db.query(ChatHistory).filter_by(user_id=user_id).delete()
    db.commit()
    return n


---## 5. Assignment & exam solvers (RAG-grounded)

In [ ]:
def _solve_assignment_llm(db, user_id: int, row, text: str, context: list[str]):
    """Parse -> plan -> (research via RAG) -> solve -> summarize for one
    assignment. Uses the user's own LLM; the LLM decides the structure."""
    llm = get_llm(db, user_id)
    if llm is None:
        empty = "\u26a0\ufe0f No Groq key configured \u2014 cannot solve."
        return empty, empty
    from langchain_core.prompts import ChatPromptTemplate
    ctx = "\n\n".join(context) if context else "(no course material available)"
    prompt = ChatPromptTemplate.from_template(
        "You are an expert academic tutor. Solve this assignment for the student.\n\n"
        "Course: {course}\n"
        "Assignment title: {title}\n"
        "Required submission format: {fmt}\n\n"
        "=== ASSIGNMENT REQUIREMENTS / DESCRIPTION ===\n{text}\n\n"
        "=== RELEVANT COURSE MATERIAL (from the student's own notes/docs) ===\n{context}\n\n"
        "Output TWO sections separated by the token <<<SUMMARY>>>:\n"
        "First section: a 2-3 sentence plain summary of what the assignment asks and what is required.\n"
        "Second section: a complete, step-by-step solution/answer to the assignment (be concrete, show work, include code/formulas where asked)."
    )
    chain = prompt | llm
    try:
        res = chain.invoke({
            "course": row.course or "General",
            "title": row.title or "Assignment",
            "fmt": (row.details or {}).get("submission_format", "Not specified"),
            "text": text or "(no description given)",
            "context": ctx,
        })
        out = (res.content or "").strip()
        if "<<<SUMMARY>>>" in out:
            summary, solution = out.split("<<<SUMMARY>>>", 1)
            return solution.strip(), summary.strip()
        return out, out[:300]
    except Exception as exc:  # noqa: BLE001
        log.warning("solve failed user=%s: %s", user_id, exc)
        return f"\u26a0\ufe0f Could not solve: {exc}", "Could not solve."


def solve_assignment(db, user_id: int, event_id: int) -> dict:
    """Public wrapper: solve + store on the assignment record."""
    row = db.query(AcademicEvent).filter_by(id=event_id, user_id=user_id, type="assignment").first()
    if row is None:
        return {"ok": False, "error": "Assignment not found"}
    text = row.description or row.title or ""
    context = rag.search_all_courses(user_id, text, k_per_course=3)
    understanding = (row.details or {}).get("understanding") or {}
    ctx_lines = []
    if understanding.get("summary"):
        ctx_lines.append(f"UNDERSTANDING SUMMARY: {understanding['summary']}")
    if understanding.get("objective"):
        ctx_lines.append(f"OBJECTIVE: {understanding['objective']}")
    for key in ("tasks", "requirements", "expected_outputs",
                "submission_requirements", "required_topics", "required_tools"):
        items = understanding.get(key) or []
        if items:
            ctx_lines.append(f"{key.replace('_', ' ').title()}: " + "; ".join(items))
    if ctx_lines:
        context = ["=== ASSIGNMENT UNDERSTANDING (analysis agent) ===", *ctx_lines] + context
    solution, summary = _solve_assignment_llm(db, user_id, row, text, context)
    details = row.details or {}
    details["summary"] = summary
    details["solution"] = solution
    row.details = details
    db.commit()
    return {"ok": True, "summary": summary, "solution": solution}


def _study_guide_llm(db, user_id: int, row):
    llm = get_llm(db, user_id)
    if llm is None:
        return "\u26a0\ufe0f No Groq key configured \u2014 cannot generate a study guide."
    from langchain_core.prompts import ChatPromptTemplate
    topics = row.topic or row.title or ""
    query = " ".join(filter(None, [row.course or "", topics])) or (row.course or "")
    # Prefer the course's own outline (uploaded doc), then any course material.
    outline_docs = rag.search_with_meta(user_id, row.course, query, k=4,
                                        where={"source": "upload"})
    docs = rag.search_with_meta(user_id, row.course, query, k=5)
    seen = {o["content"][:120] for o in outline_docs}
    merged = list(outline_docs) + [d for d in docs if d["content"][:120] not in seen]
    lines = []
    for s in merged:
        tag = (s.get("chapter") or s.get("section") or s.get("filename")
               or s.get("course") or "material")
        content = (s.get("content") or "")[:400]
        lines.append(f"[{tag}] {content}")
    ctx = "\n\n".join(lines) if lines else "(no course material or outline ingested yet)"
    prompt = ChatPromptTemplate.from_template(
        "Create a focused study guide + practice questions for the student, grounded "
        "in the attached course outline and materials below.\n\n"
        "Quiz/Exam: {title} (course: {course})\n"
        "Scope/topics stated: {topics}\n\n"
        "=== COURSE OUTLINE / MATERIAL (retrieved) ===\n{context}\n\n"
        "Provide:\n1. A study guide of the key concepts/portions to revise (based on the stated scope; "
        "explain each in plain English and note what the outline lists).\n"
        "2. 5 practice questions with brief answers, drawn from the course material.\n"
        "Never invent chapters or topics not present in the outline/material; "
        "if the scope cannot be resolved, say exactly that."
    )
    chain = prompt | llm
    try:
        res = chain.invoke({
            "title": row.title or "Quiz/Exam",
            "course": row.course or "General",
            "topics": topics or "not specified",
            "context": ctx,
        })
        return (res.content or "").strip()
    except Exception as exc:  # noqa: BLE001
        log.warning("study guide failed user=%s: %s", user_id, exc)
        return f"\u26a0\ufe0f Could not generate study guide: {exc}"


def generate_study_guide(db, user_id: int, event_id: int) -> dict:
    row = db.query(AcademicEvent).filter_by(id=event_id, user_id=user_id).first()
    if row is None or row.type not in ("quiz", "exam"):
        return {"ok": False, "error": "Event not found, or not a quiz/exam"}
    guide = _study_guide_llm(db, user_id, row)
    details = row.details or {}
    details["study_guide"] = guide
    row.details = details
    db.commit()
    return {"ok": True, "study_guide": guide}


---## 6. Missed-class reconstruction from outline

In [ ]:
def missed_summary_with_outline(db, user_id: int, course: str, date_str: str) -> dict:
    """Reconstruct a missed class by fusing three sources:
    1) course outline (ingested 'course' events / topics) — what was scheduled,
    2) Telegram messages (source pool) that day — what was actually discussed,
    3) course docs (RAG) for that week.
    The LLM decides which source to trust for each claim (CONFIRMED / COMMUNITY)."""
    llm = get_llm(db, user_id)
    if llm is None:
        return {"ok": False, "error": "No Groq key configured"}

    outline_events = academic.list_events(db, user_id, etype="course", limit=20)
    outline_topics = academic.list_events(db, user_id, etype="lecture_topic", limit=20)
    outline_lines = [e.topic or e.title for e in outline_events + outline_topics if (e.topic or e.title)]
    outline_text = "\n".join(outline_lines) if outline_lines else "(no course outline ingested)"

    stream = (
        db.query(SourcePool)
        .filter_by(user_id=user_id)
        .order_by(SourcePool.id.desc())
        .limit(60)
        .all()
    )
    msgs = [f"[{s.sender_name}] {s.text}" for s in stream]

    rag_ctx = rag.search_all_courses(user_id, f"{course} {date_str}", k_per_course=3)
    rag_text = "\n\n".join(rag_ctx) if rag_ctx else "(no course documents found)"

    from langchain_core.prompts import ChatPromptTemplate

    prompt = ChatPromptTemplate.from_template(
        "Reconstruct what a student missed in class on {date} for course '{course}'.\n\n"
        "=== COURSE OUTLINE (what was scheduled this week) ===\n{outline}\n\n"
        "=== TELEGRAM MESSAGES (what the class discussed) ===\n{messages}\n\n"
        "=== COURSE DOCUMENTS (slides/notes) ===\n{docs}\n\n"
        "Cross-reference all three and produce a summary where each claim is tagged "
        "with a confidence label: (CONFIRMED) if from the outline/official source, "
        "(COMMUNITY) if from student messages, (INFERRED) otherwise. "
        "State explicitly if a topic was scheduled in the outline. If Telegram is "
        "sparse, fall back to the outline."
    )
    chain = prompt | llm
    res = chain.invoke({
        "date": date_str,
        "course": course,
        "outline": outline_text,
        "messages": msgs or "(no Telegram messages)",
        "docs": rag_text,
    })
    summary = (res.content or "").strip()
    academic.save_missed_summary(db, user_id, course, date_str, course, summary,
                                 basis="outline+telegram+RAG")
    return {"ok": True, "summary": summary}


---## 7. Debug / verify

In [ ]:
# Quick sanity check inside the notebook:
symbols = ["get_or_create_settings", "get_decrypted_settings", "save_settings",
           "get_groq_api_key", "get_llm", "build_tools", "build_graph", "answer",
           "recent_history", "clear_history", "solve_assignment",
           "generate_study_guide", "missed_summary_with_outline"]
missing = [s for s in symbols if s not in globals()]
print("All symbols present:", not missing)
if missing:
    print("missing:", missing)
